# Azure AI Speech (Azure Speech Services)

A refresher on **Azure AI Speech** — Microsoft's unified, cloud-hosted speech platform inside Azure Cognitive Services. One resource (a *key + region*, or an Entra ID token) gives you **speech-to-text** (STT), **text-to-speech** (TTS), **speech translation**, **speaker recognition**, and **pronunciation assessment** behind a single SDK. You configure *what* you want, point it at an audio source or a string, and Azure runs the models on its infrastructure — billed per character (TTS) or per audio-hour (STT).

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes  ·  _the no-key cells (pricing math + SSML builder) run on CPU with stdlib only; the real STT/TTS calls are gated behind the `SPEECH_KEY` env var_

## 1. What & Why

**What it is.** Azure AI Speech (historically "Cognitive Services Speech", "Microsoft Speech Service") is a family of hosted speech APIs reached through one Azure resource and one SDK (`azure-cognitiveservices-speech`). The headline capabilities:

- **Speech-to-text (STT)** — real-time streaming recognition, async **batch transcription** of files in blob storage, and **Custom Speech** for fine-tuning on your domain/audio.
- **Text-to-speech (TTS)** — 400+ **neural** voices across 140+ locales, rich **SSML** (styles, roles, prosody), **Custom Neural Voice** (gated brand-voice cloning), and newer high-fidelity **HD voices**.
- **Speech translation** — speech in → translated text/speech out, in one streaming call.
- **Speaker recognition** (verification/identification) and **Pronunciation Assessment** (scores fluency/accuracy for language-learning apps).

**The problem it solves.** Running your own STT (Whisper, NeMo) or TTS (Coqui, Piper) means GPUs, model downloads, vocoder tuning, and quality that often trails the best hosted systems. Azure removes all of that: deep language coverage, one SDK with the same `SpeechConfig` object across every feature, and enterprise plumbing (Entra ID, VNet/Private Link, regional data residency, compliance) — in exchange for money and a network round-trip.

**When to reach for it.** Apps already on Azure; call-center/contact-center transcription and analytics; voicebots and IVR; accessibility/read-aloud; multilingual products that want one vendor for STT + TTS + translation; language-learning apps that need pronunciation scoring; and brand voices via Custom Neural Voice.

**When not to.** When audio cannot leave your machine (use Whisper/Piper offline); when you want absolute top-tier expressive TTS or instant arbitrary voice cloning (ElevenLabs leads, and Custom Neural Voice is gated/approval-only); or for very high-volume batch where per-unit cost dominates and a self-hosted model would be cheaper.

## 2. Mental Model

**Azure Speech is one switchboard with many lines. You authenticate once into a regional Speech *resource* (`SpeechConfig` = key + region, or a token). Then for each task you wire together three things: a `SpeechConfig` (who you are + global options like the voice or language), an `AudioConfig` (where audio comes from / goes to — mic, file, stream, default speaker), and a *recognizer* or *synthesizer* object that actually does the work.** The same `SpeechConfig` flows into STT, TTS, and translation — only the worker object changes.

```
                 your machine                                    Azure (Microsoft's infra)
 ┌────────────────────────────────────────┐   HTTPS / WebSocket   ┌──────────────────────────────┐
 │ SpeechConfig(key+region | token)        │ ───────────────────▶ │  regional Speech resource     │
 │   .speech_recognition_language="en-US"  │                      │   STT  models (real-time/batch)│
 │   .speech_synthesis_voice_name="..."    │                      │   TTS  neural voices + SSML    │
 │                                         │                      │   translation / speaker / pron │
 │ AudioConfig(mic | file | stream | spkr) │                      │                               │
 │                                         │                      │                               │
 │ SpeechRecognizer / SpeechSynthesizer    │ ◀─────────────────── │  result: text  OR  audio bytes │
 └────────────────────────────────────────┘   text  /  audio     └──────────────────────────────┘
```

Three objects decide everything:

1. **`SpeechConfig` = *who + global settings*** — credentials (`subscription`=key + `region`, or `auth_token`), plus the language for STT (`speech_recognition_language`), the voice for TTS (`speech_synthesis_voice_name`), and the output audio format.
2. **`AudioConfig` = *where audio lives*** — input from the **default mic**, a **WAV file**, or a **push/pull stream**; output to the **default speaker**, a **file**, or a **stream**. Omit it and the SDK uses the system default device.
3. **The worker = *what you're doing*** — `SpeechRecognizer` (STT), `SpeechSynthesizer` (TTS), `TranslationRecognizer`, `SpeakerRecognizer`, `PronunciationAssessment`. Call `recognize_once()` / `speak_text_async()` / start a continuous session.

## 3. Key Concepts

- **Speech *resource* (key + region).** Everything keys off a single Azure resource. You need its **key** and its **region** (e.g. `eastus`, `westeurope`) — the region is part of the endpoint, so a key from `eastus` won't work against `westeurope`. Prefer **Microsoft Entra ID** tokens over raw keys for production.
- **`SpeechConfig`.** The central config object: credentials + global options (`speech_recognition_language`, `speech_synthesis_voice_name`, `set_speech_synthesis_output_format(...)`, proxy, endpoint). Reused across STT/TTS/translation.
- **`AudioConfig`.** Where audio comes from or goes: `AudioConfig(use_default_microphone=True)`, `AudioConfig(filename="in.wav")`, `AudioConfig(use_default_speaker=True)`, or a push/pull `AudioInputStream` for custom pipelines.
- **Recognition modes.** `recognize_once_async()` for a single utterance (≤ ~15 s); **continuous recognition** (`start_continuous_recognition`) with event callbacks (`recognized`, `recognizing`, `canceled`) for long audio; and **batch transcription** (a REST API over files in blob storage) for offline bulk jobs.
- **Neural voices & SSML.** TTS voices are named `{locale}-{Name}Neural`, e.g. `en-US-JennyNeural`, `en-US-AvaMultilingualNeural`. **SSML** (`<speak>…</speak>`) adds `<voice>`, `<prosody>` (rate/pitch/volume), `<break>`, `<say-as>`, `<phoneme>`, and Azure-specific **`<mstts:express-as style="cheerful" role="...">`** for emotional styles. HD voices push naturalness further.
- **Custom Speech / Custom Neural Voice.** *Custom Speech* fine-tunes STT on your audio + transcripts/lexicon for jargon and accents. *Custom Neural Voice* trains a brand voice from recordings — **gated** (requires application/approval for responsible-AI reasons).
- **Output format.** `SpeechSynthesisOutputFormat` controls codec/sample rate (e.g. `Audio16Khz32KBitRateMonoMp3`, `Riff24Khz16BitMonoPcm`). Match it to downstream: telephony (8 kHz), web (MP3/Opus), DSP (RIFF/PCM WAV).
- **Pricing.** Metered separately per feature: **TTS ~\$15–16 per 1M characters** (neural; HD/Custom cost more), **STT ~\$1 per audio-hour** (real-time/batch standard), with a **free F0 tier** (small monthly allowance) and a paid **S0** tier. Numbers drift — confirm on the pricing page.

## 4. Setup

```bash
pip install azure-cognitiveservices-speech     # official Microsoft Speech SDK (native bindings)

# Create a Speech resource in the Azure portal (or via CLI), then export its key + region:
export SPEECH_KEY="<your-speech-resource-key>"
export SPEECH_REGION="eastus"                  # must match the resource's region
```

Create the resource in the [Azure portal](https://portal.azure.com) → *Create a resource* → **Speech** (or `az cognitiveservices account create --kind SpeechServices ...`). Grab **Key 1** and the **Region/Location** from the resource's *Keys and Endpoint* blade. The **F0 free tier** is enough to try everything; **S0** is the paid standard tier.

The SDK ships native binaries via pip — on Linux you may also need the system packages `libssl`/`libasound2` (ALSA) for microphone/speaker I/O; file-based and stream-based I/O work headless.

The runnable cells below stay **self-contained and key-free**: Example 1 reproduces the **STT + TTS pricing math**, Example 2 builds a **neural-voice SSML document** (styles + prosody) as a string, and Example 3 makes **real STT/TTS calls** gated behind `SPEECH_KEY` — so the notebook always executes top-to-bottom.

In [1]:
import sys, json

print(f"python {sys.version.split()[0]}")
print("Azure Speech = one resource (key+region) + SpeechConfig/AudioConfig + a recognizer/synthesizer.")
print("The next two cells run with NO key and NO network: pricing math, then an SSML builder.")

python 3.13.7
Azure Speech = one resource (key+region) + SpeechConfig/AudioConfig + a recognizer/synthesizer.
The next two cells run with NO key and NO network: pricing math, then an SSML builder.


## 5. Worked Examples

### Example 1 — Cost accounting: estimate STT and TTS spend before you build

Azure meters speech features **separately**: TTS is billed **per character** of input, STT is billed **per audio-hour** transcribed. Each has a small monthly **F0 free** allowance, then **S0** paid rates. Before committing to a voicebot or a transcription pipeline you want to know what each side costs at your expected volume. Pure Python, no key. (Rates drift — treat these as ballpark and confirm on the pricing page.)

In [2]:
# Approx. S0 rates + F0 monthly free allowances (verify on the Azure pricing page).
TTS = {  # text-to-speech, billed per character
    "Neural":     {"per_1m_chars": 15.0, "free_chars": 500_000},
    "HD/Neural2": {"per_1m_chars": 30.0, "free_chars": 0},
    "Custom":     {"per_1m_chars": 24.0, "free_chars": 0},   # + hosting/endpoint fees
}
STT = {  # speech-to-text, billed per audio-hour
    "RealTime":   {"per_hour": 1.0,  "free_hours": 5},
    "Batch":      {"per_hour": 1.0,  "free_hours": 5},
    "Custom":     {"per_hour": 1.40, "free_hours": 0},       # + model hosting fees
}

def tts_cost(chars, tier):
    billable = max(0, chars - TTS[tier]["free_chars"])
    return billable / 1_000_000 * TTS[tier]["per_1m_chars"]

def stt_cost(hours, tier):
    billable = max(0, hours - STT[tier]["free_hours"])
    return billable * STT[tier]["per_hour"]

# A modest voicebot: synthesize 1M chars/month of replies, transcribe 100 hours of calls.
TTS_VOLUME, STT_VOLUME = 1_000_000, 100
print(f"TTS volume: {TTS_VOLUME:,} chars/mo   |   STT volume: {STT_VOLUME} audio-hours/mo\n")

print("TEXT-TO-SPEECH")
print(f"  {'tier':<12}{'$/1M chars':>12}{'free chars':>14}{'cost':>10}")
for tier, t in TTS.items():
    print(f"  {tier:<12}{t['per_1m_chars']:>12.0f}{t['free_chars']:>14,}{tts_cost(TTS_VOLUME, tier):>10.2f}")

print("\nSPEECH-TO-TEXT")
print(f"  {'tier':<12}{'$/hour':>12}{'free hrs':>14}{'cost':>10}")
for tier, t in STT.items():
    print(f"  {tier:<12}{t['per_hour']:>12.2f}{t['free_hours']:>14}{stt_cost(STT_VOLUME, tier):>10.2f}")

# Reminder: SSML markup counts toward the TTS character bill, just like the words.
ssml_overhead = "<speak version='1.0'><voice name='en-US-JennyNeural'></voice></speak>"
print(f"\nNote: SSML tags are billed too — wrapper overhead here = {len(ssml_overhead)} chars before any words.")

TTS volume: 1,000,000 chars/mo   |   STT volume: 100 audio-hours/mo

TEXT-TO-SPEECH
  tier          $/1M chars    free chars      cost
  Neural                15       500,000      7.50
  HD/Neural2            30             0     30.00
  Custom                24             0     24.00

SPEECH-TO-TEXT
  tier              $/hour      free hrs      cost
  RealTime            1.00             5     95.00
  Batch               1.00             5     95.00
  Custom              1.40             0    140.00

Note: SSML tags are billed too — wrapper overhead here = 69 chars before any words.


### Example 2 — Build the SSML: a neural voice with an emotional style and prosody

Plain-text TTS is one line, but the real power of Azure TTS is **SSML** — and especially the Azure-specific `<mstts:express-as>` element that switches a neural voice into emotional **styles** (`cheerful`, `sad`, `newscast`, `customerservice`, …) and **roles**. Knowing the exact document shape lets you debug, port to REST, or template replies. We assemble the SSML as a string — no network call, so nothing is synthesized and no quota is spent.

In [3]:
import xml.dom.minidom as minidom

def build_ssml(text, voice="en-US-JennyNeural", style="cheerful", rate="+0%", pitch="+0%"):
    # The mstts namespace is REQUIRED for <mstts:express-as> (styles/roles).
    return (
        "<speak version='1.0' xmlns='http://www.w3.org/2001/10/synthesis' "
        "xmlns:mstts='https://www.w3.org/2001/mstts' xml:lang='en-US'>"
        f"<voice name='{voice}'>"
        f"<mstts:express-as style='{style}'>"
        f"<prosody rate='{rate}' pitch='{pitch}'>{text}</prosody>"
        "</mstts:express-as>"
        "</voice></speak>"
    )

ssml = build_ssml(
    "Your package <break time='300ms'/> arrives tomorrow. <say-as interpret-as='date'>2026-06-24</say-as>.",
    voice="en-US-AvaMultilingualNeural", style="newscast", rate="-5%", pitch="+2%",
)

print("Generated SSML (pretty-printed):\n")
print(minidom.parseString(ssml).toprettyxml(indent="  ").split("?>", 1)[1].strip())

print(f"\nbilled characters in this SSML: {len(ssml)}  (tags included!)")
print("You'd pass this to SpeechSynthesizer.speak_ssml_async(ssml) instead of speak_text_async(text).")

Generated SSML (pretty-printed):

<speak xmlns="http://www.w3.org/2001/10/synthesis" xmlns:mstts="https://www.w3.org/2001/mstts" version="1.0" xml:lang="en-US">
  <voice name="en-US-AvaMultilingualNeural">
    <mstts:express-as style="newscast">
      <prosody rate="-5%" pitch="+2%">
        Your package 
        <break time="300ms"/>
         arrives tomorrow. 
        <say-as interpret-as="date">2026-06-24</say-as>
        .
      </prosody>
    </mstts:express-as>
  </voice>
</speak>

billed characters in this SSML: 381  (tags included!)
You'd pass this to SpeechSynthesizer.speak_ssml_async(ssml) instead of speak_text_async(text).


### Example 3 — Real STT + TTS with the Speech SDK (gated)

With `azure-cognitiveservices-speech` installed and `SPEECH_KEY` / `SPEECH_REGION` set, both directions are a handful of lines and share the same `SpeechConfig`. This makes real network calls and **consumes quota**, so it's gated behind the env var. Either way the cell prints the canonical call shapes — text→audio file, and a WAV file→text recognition.

In [4]:
import os

if os.getenv("SPEECH_KEY") and os.getenv("SPEECH_REGION"):
    import azure.cognitiveservices.speech as speechsdk

    cfg = speechsdk.SpeechConfig(subscription=os.environ["SPEECH_KEY"],
                                 region=os.environ["SPEECH_REGION"])

    # --- TTS: text -> WAV file ---
    cfg.speech_synthesis_voice_name = "en-US-JennyNeural"
    audio_out = speechsdk.audio.AudioOutputConfig(filename="out.wav")
    synth = speechsdk.SpeechSynthesizer(speech_config=cfg, audio_config=audio_out)
    result = synth.speak_text_async("Hello from Azure AI Speech.").get()
    print("TTS:", result.reason, "->", len(result.audio_data), "bytes -> out.wav")

    # --- STT: recognize the WAV we just made ---
    cfg.speech_recognition_language = "en-US"
    audio_in = speechsdk.audio.AudioConfig(filename="out.wav")
    recognizer = speechsdk.SpeechRecognizer(speech_config=cfg, audio_config=audio_in)
    stt = recognizer.recognize_once_async().get()
    print("STT:", stt.reason, "->", repr(stt.text))
else:
    print("Set SPEECH_KEY and SPEECH_REGION (and `pip install azure-cognitiveservices-speech`) to run for real.\n")
    print("import azure.cognitiveservices.speech as speechsdk")
    print("cfg = speechsdk.SpeechConfig(subscription=KEY, region=REGION)\n")
    print("# TTS: text -> speaker / file")
    print("cfg.speech_synthesis_voice_name = 'en-US-JennyNeural'")
    print("synth = speechsdk.SpeechSynthesizer(speech_config=cfg,")
    print("            audio_config=speechsdk.audio.AudioOutputConfig(filename='out.wav'))")
    print("synth.speak_text_async('Hello from Azure AI Speech.').get()")
    print("# (SSML instead: synth.speak_ssml_async(ssml).get())\n")
    print("# STT: WAV file -> text")
    print("cfg.speech_recognition_language = 'en-US'")
    print("rec = speechsdk.SpeechRecognizer(speech_config=cfg,")
    print("          audio_config=speechsdk.audio.AudioConfig(filename='out.wav'))")
    print("print(rec.recognize_once_async().get().text)")

Set SPEECH_KEY and SPEECH_REGION (and `pip install azure-cognitiveservices-speech`) to run for real.

import azure.cognitiveservices.speech as speechsdk
cfg = speechsdk.SpeechConfig(subscription=KEY, region=REGION)

# TTS: text -> speaker / file
cfg.speech_synthesis_voice_name = 'en-US-JennyNeural'
synth = speechsdk.SpeechSynthesizer(speech_config=cfg,
            audio_config=speechsdk.audio.AudioOutputConfig(filename='out.wav'))
synth.speak_text_async('Hello from Azure AI Speech.').get()
# (SSML instead: synth.speak_ssml_async(ssml).get())

# STT: WAV file -> text
cfg.speech_recognition_language = 'en-US'
rec = speechsdk.SpeechRecognizer(speech_config=cfg,
          audio_config=speechsdk.audio.AudioConfig(filename='out.wav'))
print(rec.recognize_once_async().get().text)


## 6. Gotchas & Pitfalls

- **Region must match the key.** The region is baked into the endpoint. A key created in `eastus` fails against `westeurope` with an auth error — `SpeechConfig` needs the *resource's own* region, not your nearest one.
- **`recognize_once` only grabs one short utterance.** It stops at the first silence/end of segment (~15 s cap) and returns one result. For meetings, calls, or any long audio you **must** use continuous recognition (event callbacks) or batch transcription — otherwise you silently lose everything after the first phrase.
- **SSML markup is billed, and the `mstts` namespace is mandatory for styles.** Every character of SSML counts toward the TTS bill, tags included. And `<mstts:express-as>` silently does nothing (or errors) unless you declare `xmlns:mstts='https://www.w3.org/2001/mstts'` on `<speak>` — a classic "why is my voice still neutral?" bug.
- **Styles are voice-specific.** Not every neural voice supports every style/role. `cheerful`/`newscast` work on some voices and are ignored on others; check the voice's style list or the synthesis result `reason`/cancellation details instead of assuming it took.
- **Output format vs. downstream mismatch.** Default TTS output may not be what you need. Telephony wants 8 kHz µ-law/PCM; web wants MP3/Opus; DSP wants RIFF/PCM WAV. Set `set_speech_synthesis_output_format(...)` explicitly, and for STT make sure your input WAV is PCM (16 kHz/16-bit mono is the safe default) — odd codecs/sample rates cause recognition to return `NoMatch`.
- **`reason` / cancellation details are where errors hide.** Calls don't throw on failure; they return a result with `reason == Canceled` and a `cancellation_details` (error code + message: auth, quota, bad audio). Always inspect `result.reason` rather than assuming success.
- **No instant arbitrary voice cloning.** Custom Neural Voice can build a brand voice, but it's **gated** (application + responsible-AI review) and needs studio-quality recordings. If you need quick cloning from a sample, this isn't the tool.
- **F0 free tier is rate-limited.** F0 has low concurrency and monthly caps; load tests will hit `429`/throttling. Move to S0 before benchmarking real throughput.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs Azure Speech |
|---|---|---|
| **Azure AI Speech** | Azure-native apps, one vendor for STT+TTS+translation, 140+ locales, styles/SSML, pronunciation assessment, Entra ID/compliance | Cloud-only; top-tier expressive TTS trails ElevenLabs; Custom Neural Voice is gated; per-use cost |
| **Google Cloud TTS / Speech-to-Text** | GCP-native apps, broad voice catalog, telephony profiles | Tied to GCP; comparable quality; two separate products vs one unified SDK |
| **Amazon Polly / Transcribe** | AWS-native apps, neural voices, call analytics | Tied to AWS; ecosystem-dependent; split products |
| **ElevenLabs** | Best-in-class naturalness, instant voice cloning, emotive TTS | TTS-focused (weaker STT story); pricier at quality tier; not cloud-native to Azure |
| **OpenAI (TTS + Whisper API)** | Simple, cheap, decent quality in the OpenAI stack | Fewer voices, limited SSML/styles, less enterprise plumbing |
| **Whisper / NeMo / Piper / Coqui (self-hosted)** | Offline/on-device, no per-use bill, data never leaves the box | You run the ops/GPUs; no managed scaling; TTS quality trails hosted |

**Rule of thumb:** choose **Azure AI Speech when you're on Azure (or want its Entra ID/compliance/data-residency plumbing) and want one SDK covering STT, TTS, translation, speaker ID, and pronunciation scoring across many languages.** Reach for **ElevenLabs** when expressive TTS or instant cloning is the deciding factor, **Google/AWS** to stay native in those clouds, **OpenAI** for a cheap simple stack, and **Whisper/Piper/Coqui** when audio must stay offline or you're optimizing high-volume cost.

## 8. Resources

- **Azure AI Speech docs (overview + how-tos)** — https://learn.microsoft.com/azure/ai-services/speech-service/
- **Speech SDK reference (Python)** — install, classes, samples: https://learn.microsoft.com/python/api/overview/azure/cognitiveservices-speech-readme
- **Speech SSML reference** — `<voice>`, `<prosody>`, and the `mstts:express-as` styles/roles: https://learn.microsoft.com/azure/ai-services/speech-service/speech-synthesis-markup
- **Voice gallery / language & voice support** — the full neural-voice catalog per locale: https://learn.microsoft.com/azure/ai-services/speech-service/language-support
- **Batch transcription** — async STT over files in blob storage: https://learn.microsoft.com/azure/ai-services/speech-service/batch-transcription
- **Custom Neural Voice (gated)** — brand voice training + responsible-AI gating: https://learn.microsoft.com/azure/ai-services/speech-service/custom-neural-voice
- **Pricing** — per-feature rates, F0/S0 tiers, free allowances: https://azure.microsoft.com/pricing/details/cognitive-services/speech-services/
- **Samples repo (Azure-Samples/cognitive-services-speech-sdk)** — runnable end-to-end examples: https://github.com/Azure-Samples/cognitive-services-speech-sdk